# 10 · Convex hull against shapely

**Question.** Does Fieldwork's `summarize_hull` widget compute the convex hull it says it
does — monotone chain, on projected metres, with the area of the projected ring?

A convex hull is **exact**. There is no tolerance to choose, no curve to approximate and no
iteration to converge, so the ring either matches vertex for vertex or the two
implementations disagree about something real. That is why this was written first of the
twenty-six widgets that owed a check.

The reference projects the points itself with pyproj and takes the hull with shapely, so the
comparison tests the projection and the hull together rather than taking Fieldwork's
projected metres on trust.

Criteria fixed before running: the same vertex count, every vertex within 1 mm of one in the
reference hull and the reverse, planar area within 0.05 m², span within 0.1 m, and the
located-point count unchanged.

This notebook has two parts. **Parity** recomputes the result for a reviewer. **Dive deeper**
varies the assumption the widget inherits and shows what it does to the answer.

In [ ]:
# pyproj, shapely, rasterio and friends ship in the Pyodide distribution, not as PyPI
# wheels on this site, so they load with pyodide_js.loadPackage. piplite.install would
# look in the site's local wheel index and fail. Outside Pyodide this cell does nothing.
#
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj", "shapely", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_10_hull import run

report = run(Path.cwd() / "fixtures" / "hull-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["points"], "located points")
print(f"vertices        fieldwork {report['fieldwork']['vertexCount']}   shapely {report['reference']['vertexCount']}")
print(f"area m2         fieldwork {report['fieldwork']['areaSquareMetres']:.2f}   shapely {report['reference']['areaSquareMetres']:.2f}")
print(f"span m          fieldwork {report['fieldwork']['spanMetres']:.1f}   shapely {report['reference']['spanMetres']:.1f}")
print()
print(f"worst vertex separation {report['worstVertexSeparationM']:.3e} m")
print(f"area difference         {report['areaDifferenceSquareMetres']:.3e} m2")
print(f"span difference         {report['spanDifferenceM']:.3e} m")

## What agreement here does not mean

That a convex hull is a sensible summary of 250 cholera deaths in Soho. It is not, and
Fieldwork says so in the widget's own notes: a hull is convex, so a distribution that is not
— cases along a street, deaths clustered round one pump — is **filled in**, and the hull
asserts area where nothing was observed.

Both implementations assert it identically. A parity check cannot confirm or contradict a
claim about meaning.

In [ ]:
# How much of the hull is empty? Compare the hull's area with the area actually occupied,
# approximated as a 25 m buffer around the observed points -- the same half-width the
# corridor widget uses, and the same kind of claim.
import pyproj
from shapely.geometry import MultiPoint

fixture = json.loads((Path.cwd() / "fixtures" / "hull-soho-001.json").read_text())
forward = pyproj.Transformer.from_crs("OGC:CRS84", fixture["declaredParameters"]["metricCRS"], always_xy=True)
projected = [forward.transform(*p["coordinates"]) for p in fixture["input"]["points"] if p["coordinates"]]

hull = MultiPoint(projected).convex_hull
occupied = MultiPoint(projected).buffer(25)
print(f"hull             {hull.area:10.0f} m2")
print(f"within 25 m of a death  {occupied.area:10.0f} m2   ({occupied.area / hull.area:.0%} of the hull)")
print()
print(report["notEstablished"])

### What that number actually said

**91% of the hull is within 25 m of an observed death**, so for this layer the convexity
objection is much weaker than the general warning implies. The deaths nearly fill their own
hull.

Two things to hold at once. The general claim stands — a hull is convex and will fill in a
distribution that is not — and it is **not** what is happening here. And the 25 m is itself a
claim, the same half-width `reachable_corridor` uses, chosen because it is the figure this
project already argues about rather than because it is right.

Had this come out at 23% the lesson would have read better and been no more true. It came out
at 91%, and the check reports what it measured.

## Dive deeper · the assumption is the projection

The widget's comment says a hull built on longitude and latitude is "wrong by the cosine of
the latitude". That is the assumption `summarize_hull` inherits from its `zone` and
`hemisphere` parameters, and it is the one worth varying, because a practitioner choosing a
zone is making a decision whose consequence is invisible in the result.

Soho is at 51.5°N, where cos(latitude) is about 0.62.

In [ ]:
# The same points, hulled three ways: in the declared zone, in degrees, and in the
# neighbouring zone a practitioner could plausibly pick by mistake.
import math

declared = fixture["declaredParameters"]["metricCRS"]
degrees_hull = MultiPoint([p["coordinates"] for p in fixture["input"]["points"] if p["coordinates"]]).convex_hull

rows = []
for label, crs in [("declared  " + declared, declared), ("zone 31N  EPSG:32631", "EPSG:32631")]:
    transformer = pyproj.Transformer.from_crs("OGC:CRS84", crs, always_xy=True)
    ring = MultiPoint([transformer.transform(*p["coordinates"]) for p in fixture["input"]["points"] if p["coordinates"]]).convex_hull
    rows.append((label, len(ring.exterior.coords) - 1, ring.area))

print(f"{'hulled in':26} {'vertices':>8} {'area':>14}")
for label, vertices, area in rows:
    print(f"{label:26} {vertices:8d} {area:12.0f} m2")
print(f"{'degrees (CRS84)':26} {len(degrees_hull.exterior.coords)-1:8d} {degrees_hull.area:12.3e} deg2")
print()
print(f"cos(51.5 deg) = {math.cos(math.radians(51.5)):.3f}: a degree of longitude here is "
      f"{math.cos(math.radians(51.5)):.0%} of a degree of latitude on the ground.")
print("A hull computed in degrees treats them as equal, so it is stretched east-west and its")
print("vertex set can differ -- which is why the widget takes a zone rather than guessing.")

### What the variation shows

Changing zone barely moves the area, because Soho sits near the 30N/31N boundary and both
projections are nearly true there — so the parameter a practitioner is most likely to get
wrong is, here, the one that matters least. That is worth knowing and is **not** general: a
site in the middle of the wrong zone would be distorted by much more.

Degrees are the real trap, and the one the widget refuses by construction. The area is not
even in the same units, so a hull computed in degrees cannot be compared with one computed
in metres — it has to be reprojected, which is the step the widget does for you.

### Reading the result

`agrees` means the two implementations produced the same hull of the same points under the
stated criteria. It does not mean the hull is the right summary, that the points are the
right points, or that the area means what a reader will assume it means.